# 00 · Setup on Colab or Windows

Prepare Python and find a DSSAT executable so you can start the crop-modelling lessons.

## You will learn

- Install the course tools and understand the Colab DSSAT build.
- Use `detect()` and `connect()` to find a DSSAT executable.
- Explain the fresh `runs/` copy and resolve three common setup errors.

**Windows:** install Python 3.10 or newer and DSSAT 4.8 from [the DSSAT download page](https://dssat.net/downloads/) into `C:\DSSAT48`.
Run `python -m pip install "dssatlab[course]" jupyterlab ipykernel`, then download or clone [this repository](https://github.com/AbdelrahmanAmr3/dssatlab).
From the repository folder, run `cd course/00_setup` followed by `python -m jupyter lab lesson.ipynb`; use that Python kernel with its current directory set to `course/00_setup`.

**Colab:** open this notebook with its [Colab link](https://colab.research.google.com/github/AbdelrahmanAmr3/dssatlab/blob/master/course/00_setup/lesson.ipynb) and run cells from top to bottom.
The saved outputs are from Windows; Lesson 00 walks through the setup commands used together in later lessons.

In [1]:
LESSON = "00_setup"

Load the tools for tables and plots, installing the Python package first on Colab.

In [2]:
# Load the course tools in the notebook's Python environment.
import os, shutil, subprocess, sys
from pathlib import Path
if "google.colab" in sys.modules:
    package_install = subprocess.run([sys.executable, "-m", "pip", "install", "dssatlab[course]>=0.23,<0.24"], check=True, capture_output=True)
import dssatlab as dl
import matplotlib.pyplot as plt
%matplotlib inline
print("Course tools ready; Python:", sys.version.split()[0])

Course tools ready; Python: 3.14.7


🐍 Python idea: `[course]` adds pandas, matplotlib and PyYAML; `sys.executable -m pip` installs into this kernel's Python.
The `if` guard skips Colab actions on Windows, and `%matplotlib inline` keeps plots inside the notebook.

Open the lesson folder and prepare DSSAT on Colab, or use your existing Windows installation.

In [3]:
# Open the lesson folder and build or reuse DSSAT only on Colab.
if "google.colab" in sys.modules:
    if Path.cwd().name != LESSON:
        if not Path("dssatlab").exists():
            course_clone = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/AbdelrahmanAmr3/dssatlab"], check=True, capture_output=True)
        os.chdir(Path("dssatlab") / "course" / LESSON)
    installed_dssat = dl.install()
    print("Colab: repository ready; DSSAT built or reused.")
else:
    print("Windows: build skipped; use your DSSAT 4.8 installation.")

Windows: build skipped; use your DSSAT 4.8 installation.


`install()` builds the DSSAT executable and its data directory from the official source.
The first build takes a few minutes; later calls reuse that release's managed install within the same Colab runtime.
The executable and data directory are installed under `~/.cache/dssatlab/<version>` by default, or `$XDG_CACHE_HOME/dssatlab/<version>` when configured.

Inspect discovery with `detect()` to see the operating system, architecture and executable found.

In [4]:
# Display discovery fields with a filename instead of a personal path.
environment = dl.detect()
detected_dssat = environment["dssat_path"]
dl.to_dataframe([
    {"Field": "os_name", "Value": environment["os_name"], "Meaning": "Operating system"},
    {"Field": "architecture", "Value": environment["architecture"], "Meaning": "Machine architecture"},
    {"Field": "dssat_path", "Value": detected_dssat.name if detected_dssat else "None", "Meaning": "Executable found (filename only)"},
])

,Field,Value,Meaning
0,os_name,windows,Operating system
1,architecture,AMD64,Machine architecture
2,dssat_path,DSCSM048.EXE,Executable found (filename only)


`dssat_path` is a `Path` or `None`; this table shows only its filename to avoid exposing a user name.
Discovery checks saved config, `DSSAT_HOME`, platform defaults (`C:/DSSAT48` on Windows or `PATH` on Linux), then managed installs on Linux.
Finding a file does not yet prove a crop simulation will run.

Connect to DSSAT and name the folders that later lessons use for fresh inputs.

In [5]:
# Remember the DSSAT executable and show the lesson's relative folder names.
dssat = dl.connect(interactive=False)
HERE = Path.cwd()
assert HERE.name == LESSON, "Set the kernel current directory to course/00_setup, then rerun this cell."
assert (HERE / "lesson.ipynb").is_file(), "The kernel current directory must contain lesson.ipynb."
RUNS = HERE / "runs"
dl.to_dataframe([{
    "DSSAT executable (filename only)": dssat.name,
    "File exists": dssat.is_file(),
    "HERE": HERE.relative_to(HERE).as_posix(),
    "RUNS": RUNS.relative_to(HERE).as_posix(),
}])

,DSSAT executable (filename only),File exists,HERE,RUNS
0,DSCSM048.EXE,True,.,runs


`connect()` remembers the executable for later lessons; `interactive=False` reports a missing executable immediately instead of asking for input.
The kernel's current directory must be `course/00_setup`; the checks above verify the lesson folder name and that it contains `lesson.ipynb`.
With those checks passed, `.` means this lesson folder, and `runs` is the disposable folder named by `RUNS`; neither expression creates a data copy.

Preview the copy commands used in later setup cells; Lesson 00 has no data to copy.

In [6]:
# Display the shared copy commands without executing them in Lesson 00.
copy_commands = """if RUNS.exists():
    shutil.rmtree(RUNS)
_ = shutil.copytree(HERE / "data", RUNS)"""
print(copy_commands)

if RUNS.exists():
    shutil.rmtree(RUNS)
_ = shutil.copytree(HERE / "data", RUNS)


`shutil.rmtree(RUNS)` removes the previous disposable inputs and results; `copytree` copies the lesson's inputs into fresh `runs/`.
Later cells read that copy, so keep anything you want to save outside `runs/` before rerunning its setup cell.

Match these three common setup errors from [Troubleshooting](../../docs/guide/troubleshooting.md) to their next action.

In [7]:
# Show recovery instructions, not errors raised by this successful run.
dl.to_dataframe([
    {"Error": "DSSATNotFoundError", "Message (excerpt)": "DSSAT was not found", "Next action": "detect(); connect(path=...); on Linux, install()"},
    {"Error": "DSSATInstallError", "Message (excerpt)": "only supported on Linux/Colab", "Next action": "Windows: install DSSAT 4.8, then connect()"},
    {"Error": "DSSATInstallError", "Message (excerpt)": "Missing required tools on PATH", "Next action": "Install git, cmake, gfortran; retry install()"},
])

,Error,Message (excerpt),Next action
0,DSSATNotFoundError,DSSAT was not found,"detect(); connect(path=...); on Linux, install()"
1,DSSATInstallError,only supported on Linux/Colab,"Windows: install DSSAT 4.8, then connect()"
2,DSSATInstallError,Missing required tools on PATH,"Install git, cmake, gfortran; retry install()"


For missing build tools, use your system's package manager (or ask an administrator) to make the listed tools available on `PATH`, then retry `dl.install()`.
The unsupported-platform message still mentions `v0.1`; it means you should use an existing Windows DSSAT installation.

## Your turn

Choose automatic discovery or an explicit DSSAT location and check that the executable exists.

Hint: edit `chosen_path = None` to `chosen_path = "C:/DSSAT48"` on Windows, or `chosen_path = dl.detect()["dssat_path"]` on Colab, then run this cell.

In [8]:
# Choose a location and check the DSSAT executable in this cell.
chosen_path = None
chosen_dssat = dl.connect(path=chosen_path, interactive=False)
dl.to_dataframe([{
    "Choice": "automatic discovery" if chosen_path is None else "explicit path",
    "DSSAT executable (filename only)": chosen_dssat.name,
    "File exists": chosen_dssat.is_file(),
}])

,Choice,DSSAT executable (filename only),File exists
0,automatic discovery,DSCSM048.EXE,True


## Recap

- Colab installs the Python tools and builds DSSAT; Windows uses an existing DSSAT 4.8 installation.
- `detect()` inspects discovery; `connect()` chooses and remembers a DSSAT executable.
- Later setup cells create fresh `RUNS` inputs, so rerunning the setup cell removes previous results from `runs/`.

Next: [01 · Your first simulation](../01_first_simulation/lesson.ipynb).